# Model Context Protocol (MCP)

---

## What is MCP?

A communication layer that provides Claude with tools, prompts, and resources —
**without you having to write the integration code yourself**.

MCP shifts the burden of tool definitions and execution from your server to
dedicated MCP servers.

---

## Architecture
```
Our Server
    |
    MCP Client
    |
    +---> MCP Server (Tools, Prompts, Resources) ---> Outside Service (e.g. GitHub)
    |
    +---> MCP Server (Tools, Prompts, Resources) ---> Outside Service (e.g. AWS)
```

Your server acts as an **MCP Client** that connects to one or more MCP Servers.
Each MCP Server wraps an outside service and exposes its functionality.

---

## The Problem MCP Solves

Without MCP — building a GitHub chatbot requires you to author everything yourself:

| What you need | What you must write |
|---|---|
| List repositories | JSON schema + Python function |
| Get pull requests | JSON schema + Python function |
| Read issues | JSON schema + Python function |
| Manage projects | JSON schema + Python function |
| ... many more | ... for every piece of functionality |

GitHub has enormous API surface area — writing, testing, and maintaining all of
this is a significant engineering burden.

**With MCP** — a GitHub MCP Server already contains all of these tools.
You connect to it and they are immediately available to Claude.

---

## What MCP Servers Provide

Each MCP Server exposes three types of capabilities:

| Capability | Purpose |
|---|---|
| **Tools** | Functions Claude can call (equivalent to custom tool functions) |
| **Prompts** | Pre-built prompt templates for common tasks |
| **Resources** | Data sources Claude can read from |

---

## Common Questions

### Who writes MCP Servers?
Anyone can author one. Often the **service provider themselves** publishes an
official MCP Server — e.g. AWS releasing an MCP Server for their own services.

### How is MCP different from calling an API directly?
When you call an API directly, you write the tool schema and function yourself.
With MCP, those are already written inside the server — you just connect.

### Isn't MCP just tool use?
MCP and tool use are complementary but distinct. Tool use is the **mechanism**
Claude uses to call functions. MCP is about **who authors and maintains** those
functions — with MCP, someone else has already done that work for you.

---

## Key Takeaway

> MCP is not a replacement for tool use — it is a way to get pre-built,
> maintained tool integrations without writing them yourself.  
> Connect your MCP Client to an MCP Server and Claude immediately gains access
> to that service's full functionality.

# MCP — Deep Dive & Comparison

---

## The Restaurant Analogy
```
WITHOUT MCP:
You (the chef) must grow vegetables, raise animals, mill flour
— build every integration from scratch every time

WITH MCP:
You call your suppliers → they deliver ingredients → you focus on cooking

MCP is the supplier system — someone else did the hard integration work,
you just plug in and use it.
```

---

## Three Options Side by Side

### Scenario: User asks "Show me my open GitHub PRs"

---

### Option 1 — Direct API Call (you do everything)
```python
def get_github_prs(repo, token):
    response = requests.get(
        f"https://api.github.com/repos/{repo}/pulls",
        headers={"Authorization": f"Bearer {token}"},
        params={"state": "open"}
    )
    return response.json()

prs = get_github_prs("myrepo", "my_token")
```

You write the function, handle auth, parse responses, handle errors,
and maintain it whenever GitHub's API changes. Claude is not involved at all.

---

### Option 2 — Tool Use (Claude decides, you still implement)
```python
tools = [
    {
        "name": "get_pull_requests",
        "description": "Get open PRs from GitHub",
        "input_schema": {
            "type": "object",
            "properties": {"repo": {"type": "string"}}
        }
    }
]

def get_pull_requests(repo):
    return requests.get(f"https://api.github.com/repos/{repo}/pulls")

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    tools=tools,
    messages=[{"role": "user", "content": "Show my open PRs"}]
)
```

Claude decides **when** to call the tool, but you still write every schema,
every function, and handle every tool result. GitHub has 100+ endpoints —
that's 100+ tools for you to write and maintain.

---

### Option 3 — MCP (someone else already did all of that)
```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    messages=[{"role": "user", "content": "Show my open PRs"}],
    mcp_servers=[
        {
            "type": "url",
            "url": "https://mcp.github.com",   # that's it
            "name": "github"
        }
    ]
)
```

GitHub (or a third party) already wrote all 100+ tool schemas, all functions,
all auth handling, and maintains them going forward. You just plug in the URL.

---

## Who Does the Work?

| | Direct API | Tool Use | MCP |
|---|---|---|---|
| Who writes the functions? | You | You | MCP server author |
| Who writes the schemas? | You | You | MCP server author |
| Who decides when to call? | You | Claude | Claude |
| Who maintains the code? | You | You | MCP server author |

---

## The Scale Problem MCP Solves
```
Without MCP — you need to write and maintain:
  GitHub  → 100+ tools
  Slack   →  50+ tools
  Gmail   →  40+ tools
  Jira    →  80+ tools
  AWS     → 200+ tools
  Total:    470+ tools you write, test, and maintain

With MCP:
  GitHub MCP server  → plug in
  Slack MCP server   → plug in
  Gmail MCP server   → plug in
  Jira MCP server    → plug in
  AWS MCP server     → plug in
  Total you write:   0 tools
```

---

## The Reusability Problem MCP Solves
```
Without MCP:
  Developer A writes GitHub tools  ← wasted effort
  Developer B writes GitHub tools  ← wasted effort
  Developer C writes GitHub tools  ← wasted effort

With MCP:
  GitHub writes MCP server once
  Developer A, B, C all plug in   ← efficient
```

---

## MCP is NOT a Replacement for Tool Use

They work together:
```
MCP       = WHO authors and HOSTS the tools
Tool Use  = HOW Claude decides to call them
```

- Tool use = the phone call system
- MCP = the phone book (someone already listed all the numbers)

---

## Simple Mental Model
```
Direct API call  →  You cook everything from scratch
Tool use         →  You still cook, but Claude decides the menu
MCP              →  Professional kitchen delivers ready-made dishes,
                    Claude just places the order
```

---

## Key Takeaway

> MCP eliminates the integration tax.  
> Instead of every developer writing the same GitHub/Slack/AWS tools from scratch,
> someone writes the MCP Server once and everyone reuses it.  
> You just provide the URL — Claude handles the rest via the same tool use mechanism
> it already knows.

# MCP — Client Usage & Building Your Own Server

---

## Using an MCP Server as a Client

### What the MCP server contains (you don't write any of this)
```
Weather MCP Server
├── get_current_weather(city)
├── get_forecast(city, days)
└── get_humidity(city)
```

### What you (the client) write
```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "What is the weather in Mumbai right now?"}
    ],
    mcp_servers=[
        {
            "type": "url",
            "url": "https://weather-mcp-server.com/mcp",
            "name": "weather"
        }
    ]
)

print(response.content[0].text)
# → "Mumbai is currently 34°C, humid with partly cloudy skies."
```

That's all you write.

### What happens behind the scenes
```
Your Code          Claude          Weather MCP Server
    |                 |                    |
    |---"weather?"--->|                    |
    |                 |---get tools------->|
    |                 |<--tool list--------|
    |                 |---call tool------->|
    |                 |  get_current_weather("Mumbai")
    |                 |                    |---calls weather API
    |                 |<--{temp: 34}-------|
    |<--"34°C..."-----|                    |
```

You only talk to Claude. Claude talks to the MCP server.
You never touch the external API directly.

---

## Building Your Own MCP Server

### Step 1 — Install the MCP SDK
```bash
pip install mcp
```

### Step 2 — Write the Server
```python
# calculator_mcp_server.py

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("calculator")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a"""
    return a - b

@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b"""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

if __name__ == "__main__":
    mcp.run()
```

The `@mcp.tool()` decorator automatically:
- Generates the tool schema from **type hints**
- Generates the tool description from **docstrings**
- Registers the function and handles incoming calls

> Write good docstrings and type hints — Claude reads them to understand
> what each tool does.

### Step 3 — Connect to It as a Client
```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "What is 1234 multiplied by 5678?"}
    ],
    mcp_servers=[
        {
            "type": "url",
            "url": "http://localhost:8000/mcp",
            "name": "calculator"
        }
    ]
)
# → "1234 multiplied by 5678 is 7,006,652"
```

---

## How Claude Discovers Your Tools
```
Claude connects to your MCP server
         |
         v
Claude asks: "What tools do you have?"
         |
         v
Server responds with auto-generated schema:
{
  "tools": [
    {
      "name": "add",
      "description": "Add two numbers together",   ← from your docstring
      "input_schema": {
        "type": "object",
        "properties": {
          "a": {"type": "number"},                 ← from your type hints
          "b": {"type": "number"}
        }
      }
    }
  ]
}
         |
         v
User asks → Claude picks the right tool → your function runs → result returned
```

---

## Publishing Your Server

### Option A — Host it yourself (e.g. AWS EC2)
```bash
python calculator_mcp_server.py --host 0.0.0.0 --port 8000
# Anyone connects via: https://your-server.com/mcp
```

### Option B — Publish to PyPI
```toml
# pyproject.toml
[project]
name = "calculator-mcp"
version = "1.0.0"
dependencies = ["mcp"]

[project.scripts]
calculator-mcp = "calculator_mcp_server:main"
```
```bash
python -m build
twine upload dist/*

# Anyone installs and runs:
pip install calculator-mcp
calculator-mcp
```

---

## Author vs Client — Responsibilities

| | MCP Server Author | MCP Client (you) |
|---|---|---|
| Writes functions | Yes | No |
| Writes schemas | Auto-generated | No |
| Hosts the server | Yes | No |
| Maintains the code | Yes | No |
| Connects and uses | No | Just add the URL |

---

## Key Takeaway

> As an **author** — decorate functions with `@mcp.tool()`, write good docstrings
> and type hints, host the server. Everything else is automatic.  
> As a **client** — add the server URL to `mcp_servers`. That's it.  
> The docstrings and type hints you write as an author become the tool schema
> Claude reads to understand and call your tools correctly.

# MCP — Complete Communication Flow

---

## Transport Agnostic

MCP clients and servers can communicate over multiple transports:

- **stdio** (standard input/output) — most common, client and server on same machine
- **HTTP**
- **WebSockets**
- Other network protocols

---

## Core Message Types

| Message | Direction | Purpose |
|---|---|---|
| `ListToolsRequest` | Client → MCP Server | "What tools do you provide?" |
| `ListToolsResult` | MCP Server → Client | Returns list of available tools |
| `CallToolRequest` | Client → MCP Server | "Run this tool with these arguments" |
| `CallToolResult` | MCP Server → Client | Returns the result of running the tool |

---

## Complete Flow — "What repositories do I have?"
```
User          Our Server       MCP Client      MCP Server      Claude        GitHub
  |                |                |               |              |             |
  |--"What repos?->|                |               |              |             |
  |                |                |               |              |             |
  |                |--ListTools---->|               |              |             |
  |                |                |--ListTools--->|              |             |
  |                |                |<--tool list---|              |             |
  |                |<--"Here are    |               |              |             |
  |                |    the tools"  |               |              |             |
  |                |                               |              |             |
  |                |----------Query + Tools----------------------->|             |
  |                |                               |              |             |
  |                |<-----------------------ToolUse (call GitHub)--|             |
  |                |                               |              |             |
  |                |--CallTool----->|               |              |             |
  |                |                |--CallTool---->|              |             |
  |                |                |               |----request-->|             |
  |                |                |               |<---response--|             |
  |                |                |<--CallResult--|              |             |
  |                |<--"Here's the  |               |              |             |
  |                |    tool result"|               |              |             |
  |                |                               |              |             |
  |                |----------toolResult-------------------------->|             |
  |                |                               |              |             |
  |<--"Your repos--|<-------------------"Your repos are..."--------|             |
       are..."     |
```

---

## Each Component's Responsibility

| Component | Responsibility |
|---|---|
| **Our Server** | Orchestrates the flow — fetches tools, sends to Claude, handles tool calls |
| **MCP Client** | Communication bridge — abstracts protocol details from our server |
| **MCP Server** | Hosts tools, executes them, calls the external service |
| **Claude** | Decides which tool to call and when |
| **GitHub / External service** | Returns the actual data |

---

## Key Takeaway

> The MCP client abstracts away all the protocol complexity.  
> Your server just asks "give me the tools" and "run this tool" —
> the client handles the message passing, transport, and response parsing.  
> Each component has a single clear responsibility, making the system
> modular and easy to extend with new MCP servers.

# Who is the MCP Client?

---

## Simple Answer

When you use `mcp_servers=[]` in your API call, **Anthropic's servers act as the
MCP client on your behalf**. You are just the application developer.

---

## What Each Party Does
```
YOU                    ANTHROPIC SERVERS           MCP SERVER
(App Developer)        (MCP Client lives here)     (Tool Host)
───────────────        ───────────────────────     ──────────
Send message +  ─────> Receive your request
mcp_servers URL
                       Connect to MCP server ────>
                       Send ListToolsRequest ────>
                                            <───── Return tool list
                       Give tools to Claude
                       Claude picks a tool
                       Send CallToolRequest  ────>
                                            <───── Execute + return result
                       Give result to Claude
                       Claude forms answer
<──────────────────── Send final response
You get the answer
```

You never see any of the `ListToolsRequest` or `CallToolRequest` logic.
Anthropic handles all of it.

---

## The 3 Roles

| Role | Who | Responsibility |
|---|---|---|
| **App Developer** | You | Write the user-facing application, call Anthropic API |
| **MCP Client** | Anthropic's servers | Handle all MCP protocol communication |
| **MCP Server** | Third-party tool host | Host and execute the tools |

---

## When Would YOU Be the MCP Client?

Only if you manually handle the MCP protocol yourself — without `mcp_servers=[]`:
```python
from mcp import ClientSession
from mcp.client.sse import sse_client

async with sse_client("https://weather-mcp.com/mcp") as (read, write):
    async with ClientSession(read, write) as session:

        # YOU manually send ListToolsRequest
        tools = await session.list_tools()

        # YOU manually send CallToolRequest
        result = await session.call_tool("get_weather", {"city": "Mumbai"})

        # YOU manually pass results back to Claude
        response = client.messages.create(...)
```

This is complex, low-level work. **99% of the time you don't need this.**

---

## Two Approaches Compared
```
mcp_servers=[] approach (recommended):
You → Anthropic (does all MCP client work) → MCP Server

Manual MCP client approach:
You (do all MCP client work yourself) → MCP Server → You → Anthropic
```

---

## Key Takeaway

> Just use `mcp_servers=[]` and let Anthropic be the MCP client for you.  
> You only need to become the MCP client yourself if you have a very specific
> reason to control the MCP communication at the protocol level.

# MCP Project — CLI Chatbot with MCP Client + Server

---

## What We're Building

A CLI-based chatbot that lets users interact with a collection of documents.
```
User (terminal)
      |
      v
MCP Client (main.py / mcp_client.py)
      |
      v
MCP Server (mcp_server.py) — provides tools to read/update documents
      |
      v
Claude — uses the tools to answer user questions
```

---

## Two Components

| Component | File | Responsibility |
|---|---|---|
| **MCP Client** | `mcp_client.py` | Handles user interaction, talks to Claude and MCP server |
| **MCP Server** | `mcp_server.py` | Hosts document tools, stores documents in memory |

### Tools the server will provide:
- `read_document(name)` — read a document's contents
- `update_document(name, content)` — update a document's contents

Documents are stored **in memory** — no database needed.

---

## Important Architecture Note

In real-world projects you typically build **one side**, not both:

| If you are... | You build... |
|---|---|
| Exposing your service to others | MCP Server only |
| Connecting to existing services | MCP Client only |

We are building both sides here **purely for learning** — to see exactly how
the two sides communicate with each other.

---

## Project Setup
```
1. Extract cli_project.zip to your project directory
2. Add your Anthropic API key to the .env file
3. Install dependencies
4. Run the app to verify setup
```

### Install and run
```bash
# Using UV (recommended)
uv run main.py

# Using standard Python
python main.py
```

### Verify it works
```
> what's 1+1?
→ Claude responds: "2"
```

If you get a response, the base setup is working correctly.

---

## Project File Structure
```
cli_project/
├── main.py          # Entry point
├── mcp_client.py    # MCP client logic
├── mcp_server.py    # MCP server + tool definitions
├── .env             # API key goes here
└── README.md        # Full setup instructions
```

---

## Key Takeaway

> This project exists to make the MCP client/server communication concrete —
> you will see both sides of the protocol in your own code.  
> In production, you would only build the side relevant to your use case.

# Building an MCP Server with the Python SDK

---

## Setup
```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")
```

One line to create a fully functional MCP server.

---

## In-Memory Document Store
```python
docs = {
    "deposition.md":  "This deposition covers the testimony of Angela Smith, P.E.",
    "report.pdf":     "The report details the state of a 20m condenser tower.",
    "financials.docx":"These financials outline the project's budget and expenditure",
    "outlook.pdf":    "This document presents the projected future performance...",
    "plan.md":        "The plan outlines the steps for the project's implementation.",
    "spec.txt":       "These specifications define the technical requirements..."
}
```

Simple Python dict — keys are document IDs, values are content strings.

---

## Tool 1 — Read Document
```python
@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string."
)
def read_document(
    doc_id: str = Field(description="Id of the document to read")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    return docs[doc_id]
```

---

## Tool 2 — Edit Document
```python
@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string in the document's content with a new string."
)
def edit_document(
    doc_id: str = Field(description="Id of the document that will be edited"),
    old_str: str = Field(description="The text to replace. Must match exactly, including whitespace."),
    new_str: str = Field(description="The new text to insert in place of the old text.")
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")

    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```

---

## How the SDK Works
```
@mcp.tool decorator
      |
      v
Reads your Python type hints  →  generates JSON schema automatically
Reads your Field() descriptions →  populates tool description for Claude
Reads your docstring/name     →  registers the tool by name
```

You write Python. The SDK handles the MCP protocol details.

---

## Manual (Without SDK) vs SDK Approach

| | Without SDK | With SDK |
|---|---|---|
| Schema | Write full JSON by hand | Auto-generated from type hints |
| Descriptions | Manually in JSON | `Field(description="...")` |
| Registration | Manual | `@mcp.tool` decorator |
| Validation | DIY | Pydantic built-in |
| Boilerplate | High | Minimal |

---

## Error Handling

Both tools raise `ValueError` with descriptive messages when a document ID
does not exist. Claude receives the error message and can act on it —
e.g. telling the user the document was not found or trying a different ID.

---

## Key Takeaway

> The MCP Python SDK lets you focus on **business logic** — the SDK generates
> the JSON schemas, handles registration, and manages protocol details automatically.  
> Type hints define the schema. `Field()` descriptions guide Claude.
> `@mcp.tool` does the rest.

# Testing MCP Servers with the MCP Inspector

---

## What It Is

A built-in browser-based tool from the Python MCP SDK for testing and debugging
your MCP server in isolation — no need to wire up Claude or a full application.

---

## Starting the Inspector
```bash
# Make sure your environment is activated first, then:
mcp dev mcp_server.py
```

This starts a development server on port 6277 and gives you a local URL to open
in your browser.

> The inspector UI is actively being developed — the interface may look different
> from screenshots, but core functionality stays consistent.

---

## Inspector Interface

Once open, click **Connect** to start your MCP server. The nav bar shows:

- Resources
- Prompts
- **Tools** ← main area for testing

---

## Testing a Tool
```
1. Navigate to Tools section
2. Click "List Tools" → see all registered tools
3. Select a tool → testing interface opens
4. Fill in the required parameters
5. Click "Run Tool" → see result or error
```

**Example — test `read_doc_contents`:**
```
doc_id: "deposition.md"
→ Returns: "This deposition covers the testimony of Angela Smith, P.E."
```

**Chain operations to verify edits:**
```
1. Run edit_document → replace some text
2. Run read_doc_contents on same doc → confirm change was applied
```

---

## Development Workflow
```
Edit mcp_server.py
       |
       v
Test tool in inspector
       |
       v
Verify result / debug error
       |
       v
Repeat — no full app setup needed
```

---

## Key Takeaway

> The MCP inspector gives you a fast feedback loop during server development.  
> Test each tool individually, verify results, and debug in isolation
> before connecting your server to Claude or any client application.

# Building the MCP Client

---

## Architecture
```
CLI Application (main.py)
        |
        v
MCP Client (our custom class)    ← wraps the session, handles cleanup
        |
        v
Client Session (MCP SDK)         ← actual connection to the MCP server
        |
        v
MCP Server (mcp_server.py)       ← hosts the tools
```

---

## Two Components

| Component | What it is | Responsibility |
|---|---|---|
| **MCP Client** | Our custom class | Wraps the session, exposes simple methods, handles resource cleanup |
| **Client Session** | MCP SDK built-in | The actual low-level connection to the server |

---

## What the Client Needs to Do

The application only needs two capabilities from the MCP client:

1. **Get available tools** → send to Claude with the user's message
2. **Execute a tool** → when Claude requests one, run it and return the result

---

## Implementation

### `list_tools()` — get all tools from the server
```python
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools
```

### `call_tool()` — execute a specific tool
```python
async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)
```

Both methods delegate directly to the SDK's `ClientSession` — the custom class
just makes them easy to call and handles connection lifecycle.

---

## Testing the Client
```python
async with MCPClient(
    command="uv", args=["run", "mcp_server.py"]
) as client:

    # Test list_tools
    tools = await client.list_tools()
    print(tools)
    # → [read_doc_contents, edit_document]

    # Test call_tool
    result = await client.call_tool(
        "read_doc_contents", {"doc_id": "report.pdf"}
    )
    print(result)
    # → "The report details the state of a 20m condenser tower."
```

---

## Complete Application Flow
```
User asks: "What is in report.pdf?"
        |
        v
1. client.list_tools()          → get [read_doc_contents, edit_document]
        |
        v
2. Send tools + question to Claude
        |
        v
3. Claude responds: call read_doc_contents(doc_id="report.pdf")
        |
        v
4. client.call_tool("read_doc_contents", {"doc_id": "report.pdf"})
        |
        v
5. Result sent back to Claude
        |
        v
6. Claude responds to user in plain English
```

---

## Key Takeaway

> The MCP client is just a thin wrapper around the SDK's `ClientSession`.  
> It exposes two methods — `list_tools()` and `call_tool()` — which is all the
> application needs to integrate fully with any MCP server.  
> The custom class exists mainly to handle session lifecycle and cleanup cleanly.

# MCP Resources

---

## What Resources Are

Resources allow the MCP Server to **expose data to the client** — similar to GET
request handlers in an HTTP server. They fetch data rather than perform actions.

| MCP Concept | HTTP Equivalent |
|---|---|
| Resource | GET endpoint |
| Tool | POST endpoint |

---

## Use Case — Document @Mentions

When a user types `@` in the CLI:
```
User types "@"
       |
       v
Our Code sends ReadResourceRequest("docs://documents") to MCP Server
       |
       v
MCP Server returns list of document names
       |
       v
Autocomplete dropdown shows the list
```

When a user mentions `@report.pdf` in a message:
```
Our Code sends ReadResourceRequest("docs://documents/report.pdf")
       |
       v
MCP Server returns document contents
       |
       v
Our Code injects contents into the prompt as XML:
<document id="report.pdf">
...condition of a 20m condenser tower...
</document>
```

---

## Two Types of Resources

### Direct Resource — static URI, no parameters
```python
@mcp.resource(
    "docs://documents",           # fixed URI
    mime_type="application/json"
)
def list_docs() -> list[str]:
    return list(docs.keys())      # returns ["deposition.md", "report.pdf", ...]
```

### Templated Resource — URI with parameters
```python
@mcp.resource(
    "docs://documents/{doc_id}",  # {doc_id} is extracted from URI automatically
    mime_type="text/plain"
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]
```

The Python SDK parses `{doc_id}` from the URI and passes it as a function argument.

---

## MIME Types

The `mime_type` field tells the client what kind of data to expect:

| MIME type | Use for |
|---|---|
| `application/json` | Structured list or dict data |
| `text/plain` | Plain text document content |
| Any valid MIME type | Binary, HTML, CSV, etc. |

The SDK handles serialization automatically — return a Python object, not a string.

---

## Resource Message Types

| Message | Direction | Purpose |
|---|---|---|
| `ReadResourceRequest` | Client → MCP Server | Request data at a given URI |
| `ReadResourceResult` | MCP Server → Client | Returns the resource data |

---

## Testing in MCP Inspector
```bash
uv run mcp dev mcp_server.py
```

In the inspector you will see:
- **Resources** tab — shows direct/static resources
- **Resource Templates** tab — shows templated resources with parameters

Click any resource, fill in parameters if needed, and run to see the exact
response your client will receive.

---

## Key Takeaways

- Resources **expose data**; tools **perform actions**
- Direct resources → fixed URI, no params
- Templated resources → URI params become function arguments
- `mime_type` tells the client how to interpret the response
- SDK handles serialization — return Python objects directly

# MCP Resources — Client Side

---

## What the Client Needs to Do

When a user mentions `@report.pdf`, the application must:
```
1. Call read_resource("docs://documents/report.pdf") on the MCP client
2. MCP client sends ReadResourceRequest to MCP server
3. MCP server returns document contents
4. Application injects contents directly into the Claude prompt
```

No tool call needed — the content goes straight into the prompt.

---

## Required Imports
```python
import json
from pydantic import AnyUrl
```

- `AnyUrl` — ensures proper type handling for the URI parameter
- `json` — parses JSON responses from the server

---

## Implementing `read_resource()`
```python
async def read_resource(self, uri: str) -> Any:
    result = await self.session().read_resource(AnyUrl(uri))
    resource = result.contents[0]       # first item contains the data

    if isinstance(resource, types.TextResourceContents):
        if resource.mimeType == "application/json":
            return json.loads(resource.text)    # parse JSON into Python object

        return resource.text                    # return plain text as-is
```

---

## Response Structure
```
ReadResourceResult
    └── contents[0]
            ├── mimeType   →  "application/json" or "text/plain"
            └── text       →  the actual data as a string
```

MIME type tells you how to parse the data — JSON gets `json.loads()`,
plain text is returned directly.

---

## How Resources Flow Into Prompts
```python
# User types: "What's in the @report.pdf document?"
# Application detects mention, fetches resource, injects into prompt:

prompt = f"""Answer the user's query:
<query>
{user_message}
</query>

The user may have referenced a document. Here is its contents:
<document id="report.pdf">
{await client.read_resource("docs://documents/report.pdf")}
</document>
"""
```

Claude receives the document content directly — no tool call round-trip needed.

---

## Resource vs Tool — When to Use Which

| Scenario | Use |
|---|---|
| Inject document contents into prompt before Claude responds | Resource |
| Let Claude decide when to look up a document | Tool |
| Autocomplete list of available documents | Resource |
| Claude modifying a document | Tool |

---

## Key Takeaway

> `read_resource()` is the client-side counterpart to `@mcp.resource()` on the server.  
> It fetches data and returns it as a Python object — the application then decides
> how to use it, typically by injecting it directly into the prompt sent to Claude.  
> This avoids a tool call round-trip and makes interactions faster.

# MCP Prompts

---

## What Prompts Are

Prompts let the MCP Server expose **pre-built, high-quality message templates**
that clients can use instead of writing their own prompts from scratch.

They represent the MCP author's expertise — carefully crafted and tested so users
get better results than they could achieve themselves.

---

## Why This Matters

**User writes:** `"Convert report.pdf to markdown"`
→ Works, but produces mediocre results

**Prompt from MCP server:** A thoroughly eval'd prompt with detailed instructions,
structure guidelines, output format requirements, and step-by-step analysis
→ Produces consistently high-quality results

The MCP server author does the prompt engineering once — every user benefits.

---

## How Prompts Work

A prompt function returns a **list of messages** that the client can send
directly to Claude. The decorator registers the prompt by name.
```python
from mcp.server.fastmcp import base
from pydantic import Field

@mcp.prompt(
    name="format",
    description="Rewrites the contents of a document in Markdown format."
)
def format_document(
    doc_id: str = Field(description="Id of the document to format")
) -> list[base.Message]:

    prompt = f"""
You are a document conversion specialist tasked with rewriting documents
in Markdown format. Your goal is to take the content of a given document
and convert it into well-structured Markdown, preserving the original
meaning and enhancing readability.

Here is the identifier of the document you need to convert:
<document id>
{doc_id}
</document id>

Instructions:
1. Retrieve the content of the document associated with the given document_id.
2. Analyze the structure and content of the document.
3. Convert the document to Markdown format, following these guidelines:
   - Use appropriate header levels (# for main titles, ## for subtitles)
   - Properly format lists (both ordered and unordered)
   - Use emphasis (*italic* or **bold**) where appropriate
   - Add links and images using Markdown syntax if present

Before providing the final Markdown output, in <document analysis> tags:
   - Identify the main sections and subsections
   - Count the number of sections to ensure proper nesting of headers

Please proceed with your analysis and conversion.
"""
    return [base.UserMessage(prompt)]
```

---

## User Flow — /format Command
```
User types "/"
       |
       v
CLI shows autocomplete: /format  "Rewrites the conten..."
       |
       v
User selects /format and provides doc_id (e.g. report.pdf)
       |
       v
Client calls GetPrompt("format", {doc_id: "report.pdf"})
       |
       v
MCP Server returns list of messages with interpolated prompt
       |
       v
Client sends messages to Claude
       |
       v
Claude reads the document, reformats it, uses edit_document tool to save
```

---

## Testing in MCP Inspector
```bash
uv run mcp dev mcp_server.py
```

Navigate to the **Prompts** section → select your prompt → fill in parameters
→ run to see the exact messages that would be sent to Claude.

---

## Three MCP Server Capabilities — Summary

| Capability | Decorator | Purpose |
|---|---|---|
| **Tools** | `@mcp.tool()` | Perform actions (read, write, call APIs) |
| **Resources** | `@mcp.resource()` | Expose data (fetch documents, lists) |
| **Prompts** | `@mcp.prompt()` | Provide pre-built, high-quality message templates |

---

## Best Practices

- Focus prompts on tasks **central to your server's purpose**
- Write **detailed, specific instructions** — not vague requests
- Test thoroughly with different inputs before publishing
- Write clear `description` fields so clients know when to use each prompt
- Design prompts to work naturally with your server's tools and resources

---

## Key Takeaway

> Prompts are the MCP server author's eval'd, production-ready templates.  
> They return a list of `base.Message` objects ready to be sent to Claude.  
> The value is in the prompt engineering — users get expert-level instructions
> without having to write them.

# MCP Prompts — Client Side

---

## Two Methods to Implement

### `list_prompts()` — get all available prompts
```python
async def list_prompts(self) -> list[types.Prompt]:
    result = await self.session().list_prompts()
    return result.prompts
```

Returns all prompts registered on the MCP server — used to populate the `/`
autocomplete in the CLI.

---

### `get_prompt()` — fetch a specific prompt with arguments
```python
async def get_prompt(self, prompt_name: str, args: dict[str, str]):
    result = await self.session().get_prompt(prompt_name, args)
    return result.messages    # list of messages ready to send to Claude
```

The `args` dict maps parameter names to values. The MCP server passes them as
keyword arguments to the prompt function, interpolating them into the template.

---

## How Arguments Flow
```
Client calls:
    get_prompt("format", {"doc_id": "report.pdf"})
         |
         v
MCP Server calls:
    format_document(doc_id="report.pdf")
         |
         v
Server returns:
    [UserMessage("...reformat the document with id: report.pdf...")]
         |
         v
Client sends messages directly to Claude
```

---

## CLI Workflow — /format Command
```
1. User types "/"
         |
         v
2. client.list_prompts() → shows [/format, ...]
         |
         v
3. User selects /format → prompted to pick a doc_id
         |
         v
4. client.get_prompt("format", {"doc_id": "report.pdf"})
         |
         v
5. Returns list of messages → sent to Claude
         |
         v
6. Claude reads prompt, uses read_doc_contents tool,
   reformats content, uses edit_document tool to save
```

---

## MCP Client — All Methods Summary

| Method | What it calls | Returns |
|---|---|---|
| `list_tools()` | `session.list_tools()` | List of tool definitions |
| `call_tool()` | `session.call_tool()` | Tool execution result |
| `read_resource()` | `session.read_resource()` | Resource data (text/JSON) |
| `list_prompts()` | `session.list_prompts()` | List of prompt definitions |
| `get_prompt()` | `session.get_prompt()` | List of messages for Claude |

---

## Key Takeaway

> `list_prompts()` populates your command palette.  
> `get_prompt()` fetches the interpolated messages ready to send directly to Claude.  
> The client never constructs the prompt itself — it just passes arguments to the
> server and forwards the returned messages to Claude.

# MCP Server Primitives — Summary

---

## The Three Primitives

| | Tools | Resources | Prompts |
|---|---|---|---|
| **Controlled by** | Claude (model) | Our app | The user |
| **Who decides when to use** | Claude decides autonomously | App decides when to fetch | User triggers explicitly |
| **Results used by** | Claude | Our app | Claude (via our app) |
| **Used for** | Giving Claude additional functionality | Getting data into the app, adding context to messages | Workflows triggered by user input — slash commands, button clicks, menu options |

---

## Mental Model
```
Tools     →  Claude's hands    (Claude reaches out to do things)
Resources →  App's eyes        (App fetches data to see and use)
Prompts   →  User's shortcuts  (User triggers pre-built workflows)
```

---

## Real Examples — Document Chatbot

### Tools
```python
@mcp.tool()
def edit_document(doc_id: str, old_str: str, new_str: str):
    """Edit a document by replacing text"""
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
```
Claude autonomously calls this when it decides it needs to modify a document.
The user never directly triggers it.

---

### Resources
```python
@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    return list(docs.keys())   # app uses this to populate @mention autocomplete

@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    return docs[doc_id]        # app injects this into the prompt when user types @report.pdf
```
Our app calls these — not Claude. Results are used to build the UI or enrich prompts.

---

### Prompts
```python
@mcp.prompt(name="format", description="Rewrites a document in Markdown format.")
def format_document(doc_id: str) -> list[base.Message]:
    return [base.UserMessage(f"Reformat {doc_id} using proper markdown headers, lists, and tables...")]
```
User types `/format report.pdf` → app fetches this prompt → sends to Claude.
The user consciously chose to run this workflow.

---

## Decision Guide — Which Primitive to Use?

| Situation | Use |
|---|---|
| Claude needs to perform an action (edit, create, call API) | Tool |
| Claude needs to decide when to look something up | Tool |
| App needs a list of documents for autocomplete | Resource |
| App needs to inject document content into a prompt | Resource |
| User wants to trigger a complex workflow with one command | Prompt |
| User wants to run a well-tested, pre-built instruction set | Prompt |

---

## Key Takeaway

> Tools give Claude agency.  
> Resources give your app data.  
> Prompts give users power.  
> Together they cover everything an MCP server needs to expose.